In [ ]:
import gplately
import gplately.pygplates as pygplates
import numpy as np
import matplotlib.pyplot as plt
import os, glob, joblib
os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"

In [ ]:
os.path.join(os.environ.get("CCD_PLATE_MODEL_ROOT", "."),
             "Alfonso_etal_2024_modClennettMuller")

In [ ]:


# Method 1: manually point to files
model_dir = "./Alfonso_etal_2024_modClennettMuller/"

feature_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/"
        r"*.gpml",
    )
)

#files = glob.glob("/my/data/folder/**/*.gpml", recursive=True)


rotation_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/",
        r"*.rot",
    )
)
coastlines_filename = os.path.join(
    model_dir,
    "Coastlines",
    "Clennett__etal_2020_Coastlines.gpml",
)

static_polygons = os.path.join(
    model_dir,
    "StaticPolygons/Clennett_2020_StaticPolygons.gpml"
)

reconstruction = gplately.PlateReconstruction(
    rotation_model=rotation_filenames,
    topology_features=pygplates.FeatureCollection(
        [
            i for i in pygplates.FeaturesFunctionArgument(
                feature_filenames
            ).get_features()
            if i.get_feature_type().to_qualified_string()
            != "gpml:TopologicalSlabBoundary"
        ]
        
    ),
    static_polygons=static_polygons
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=reconstruction,
    continents=coastlines_filename,
)



In [ ]:
seafloorgrid = gplately.SeafloorGrid(
    reconstruction,
    gplot,
    170,
    0,
    1,
    save_directory="./",
    file_collection='Alfonso2024',
    refinement_levels = 6,

    initial_ocean_mean_spreading_rate = 100.0,
    resume_from_checkpoints=True,
    zval_names= ['SPREADING_RATE'],
)

seafloorgrid.reconstruct_by_topologies()


In [ ]:
seafloorgrid.lat_lon_z_to_netCDF("SEAFLOOR_AGE", unmasked=True, nprocs=-1)
seafloorgrid.lat_lon_z_to_netCDF("SPREADING_RATE", unmasked=True, nprocs=-1)

In [ ]:
gplately.Raster(
    './SEAFLOOR_AGE/Muller2019_SEAFLOOR_AGE_grid_1.00Ma.nc'
).imshow()

In [ ]:
gplately.Raster(
os.path.join(os.environ.get("CCD_MASK_ROOT", "continent_masks"),
             "AGEGRIDDING/continent_mask/Muller2019_continent_mask_0.00Ma.nc")
).data

In [ ]:
gplately.Raster(os.path.join(os.environ.get("CCD_MASK_ROOT", "continent_masks"),
             "CONTOURS/Sep17-Muller2022/continent_mask_0.0.nc")).data

In [ ]:
gplately.Raster(os.path.join(os.environ.get("CCD_MASK_ROOT", "continent_masks"),
             "CONTOURS/Sep19-Muller2022/continent_mask_0.0.nc")).data